In [ ]:
import os
import random
import shutil
from sklearn.model_selection import StratifiedShuffleSplit

image_dir = "/home/jonggyun/jupyter_home/yolo8n_tank_image/image"
base_dir = "/home/jonggyun/jupyter_home/yolo8n_tank_image/yolo8n_tank_image_dataset"
save_dir_40 = "/home/jonggyun/jupyter_home/yolo8n_tank_image/yolov8n40"

# 기존 폴더 초기화
for folder in ['train', 'validation']:
    path = os.path.join(base_dir, folder)
    if os.path.exists(path):
        shutil.rmtree(path)
if os.path.exists(save_dir_40):
    shutil.rmtree(save_dir_40)

os.makedirs(os.path.join(save_dir_40, "images"), exist_ok=True)
os.makedirs(os.path.join(save_dir_40, "labels"), exist_ok=True)

# 전체 이미지 리스트
all_images = [f for f in os.listdir(image_dir) if f.lower().endswith(('.png', '.jpg'))]

# 라벨 있는 이미지와 없는 이미지 분리
labeled_images = []
labels = []
unlabeled_images = []

for img in all_images:
    label_path = os.path.join(image_dir, os.path.splitext(img)[0] + '.txt')
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            lines = f.readlines()
        if lines:  # 라벨 파일에 내용 있음
            labeled_images.append(img)
            # 첫 줄의 클래스 번호
            class_id = int(lines[0].split()[0])
            labels.append(class_id)
        else:
            # 빈 라벨 파일(내용 없음)
            unlabeled_images.append(img)
    else:
        # 라벨 파일 없음인 경우도 unlabeled_images에 포함 가능 (필요시)
        unlabeled_images.append(img)

# 1) 라벨 있는 이미지만 stratified split 60% / 40%
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.4, random_state=42)
train_val_idx, test_idx = next(sss.split(labeled_images, labels))

labeled_60 = [labeled_images[i] for i in train_val_idx]
labels_60 = [labels[i] for i in train_val_idx]

labeled_40 = [labeled_images[i] for i in test_idx]

# 2) 라벨 있는 60%를 다시 stratified split 80% train / 20% val
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(sss2.split(labeled_60, labels_60))

train60 = [labeled_60[i] for i in train_idx]
val60 = [labeled_60[i] for i in val_idx]

# 3) 빈 라벨 이미지를 전체 비율(60% train/val, 40% test)에 맞게 랜덤 배분
random.seed(42)
random.shuffle(unlabeled_images)

num_unlabeled_40 = int(0.4 * len(unlabeled_images))
unlabeled_40 = unlabeled_images[:num_unlabeled_40]
unlabeled_60 = unlabeled_images[num_unlabeled_40:]

# unlabeled 60%를 다시 80% train / 20% val
num_unlabeled_train = int(0.8 * len(unlabeled_60))
unlabeled_train = unlabeled_60[:num_unlabeled_train]
unlabeled_val = unlabeled_60[num_unlabeled_train:]

# 4) 최종 train/val/test 리스트 병합
final_train = train60 + unlabeled_train
final_val = val60 + unlabeled_val
final_test = labeled_40 + unlabeled_40

# 5) 파일 복사 함수
def copy_files(image_list, img_dst, lbl_dst, src_dir=image_dir):
    os.makedirs(img_dst, exist_ok=True)
    os.makedirs(lbl_dst, exist_ok=True)
    for img in image_list:
        label = os.path.splitext(img)[0] + '.txt'
        shutil.copy(os.path.join(src_dir, img), os.path.join(img_dst, img))
        shutil.copy(os.path.join(src_dir, label), os.path.join(lbl_dst, label))

# 6) 복사 수행
copy_files(final_train, os.path.join(base_dir, 'train/images'), os.path.join(base_dir, 'train/labels'))
copy_files(final_val, os.path.join(base_dir, 'validation/images'), os.path.join(base_dir, 'validation/labels'))
copy_files(final_test, os.path.join(save_dir_40, "images"), os.path.join(save_dir_40, "labels"))

print(f"✅ train: {len(final_train)}개, val: {len(final_val)}개, test(40%): {len(final_test)}개")

In [ ]:
import os
from collections import Counter

def get_class_distribution(image_list, image_dir):
    class_counts = Counter()
    total_with_label = 0
    total_without_label = 0

    for img in image_list:
        label_path = os.path.join(image_dir, os.path.splitext(img)[0] + '.txt')
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                lines = f.readlines()
            if lines:
                for line in lines:
                    class_id = int(line.split()[0])
                    class_counts[class_id] += 1
                total_with_label += 1
            else:
                total_without_label += 1
        else:
            total_without_label += 1

    total_labeled = sum(class_counts.values())
    print(f"총 이미지 수: {len(image_list)}")
    print(f"라벨 있는 이미지 수: {total_with_label}")
    print(f"빈 라벨 또는 라벨 없음 이미지 수: {total_without_label}")
    print("클래스별 객체 수 및 비율:")
    for cls, cnt in class_counts.items():
        ratio = cnt / total_labeled * 100 if total_labeled > 0 else 0
        print(f" 클래스 {cls}: {cnt}개 ({ratio:.2f}%)")
    print('-' * 30)

# 예시 사용법
print("=== Train 데이터 클래스 분포 ===")
get_class_distribution(final_train, image_dir)

print("=== Validation 데이터 클래스 분포 ===")
get_class_distribution(final_val, image_dir)

print("=== Test(40%) 데이터 클래스 분포 ===")
get_class_distribution(final_test, image_dir)

In [ ]:
train: '/home/jonggyun/jupyter_home/yolo8n_tank_image/yolo8n_tank_image_dataset/train/images'
val: '/home/jonggyun/jupyter_home/yolo8n_tank_image/yolo8n_tank_image_dataset/validation/images'

nc: 3
names: ['E_Tank', 'Car', 'Human']

In [ ]:
data_yaml_content = """
train: /home/jonggyun/jupyter_home/yolo8n_tank_image/yolo8n_tank_image_dataset/train/images
val: /home/jonggyun/jupyter_home/yolo8n_tank_image/yolo8n_tank_image_dataset/validation/images

nc: 3
names: ['E_Tank', 'Car', 'Human']
"""

with open("/home/jonggyun/jupyter_home/yolo8n_tank_image/yolo8n_tank_image_dataset/data.yaml", "w") as f:
    f.write(data_yaml_content)

print("✅ 3개 클래스 포함한 data.yaml 생성 완료")

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # 또는 yolov8s.pt 등

model.train(
    data='/home/jonggyun/jupyter_home/yolo8n_tank_image/yolo8n_tank_image_dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    name='multi_class_yolov8',
    project='/home/jonggyun/jupyter_home/yolo8n_tank_image',
    augment=True
    
)

In [ ]:
model = YOLO('/home/jonggyun/jupyter_home/yolo8n_tank_image/multi_class_yolov8/weights/best.pt')

results = model.predict(source='/home/jonggyun/jupyter_home/yolo8n_tank_image/yolo8n_tank_image_dataset/validation/images', save=True)

In [ ]:
from ultralytics import YOLO
import os

# 모델 로드
model = YOLO('/home/jonggyun/jupyter_home/yolo8n_tank_image/multi_class_yolov8/weights/best.pt')

# 테스트 이미지 경로
test_img_dir = '/home/jonggyun/jupyter_home/yolo8n_tank_image/extest_images'

# 이미지 예측 수행
results = model.predict(
    source=test_img_dir,
    save=True,          # 결과 이미지 저장
    save_txt=True,      # 감지 결과를 txt로 저장
    imgsz=640,
    conf=0.25
)

# 예측 완료 메시지 출력
print("✅ 테스트 이미지 예측 완료")

In [ ]:
import os

test_img_dir = "/home/jonggyun/jupyter_home/yolo8n_tank_image/extest_images"
os.makedirs(test_img_dir, exist_ok=True)
print(f"✅ 폴더 생성 완료: {test_img_dir}")

In [ ]:
from ultralytics import YOLO

model_path = '/home/jonggyun/jupyter_home/yolo8n_tank_image/multi_class_yolov8/weights/best.pt'
test_img_dir = '/home/jonggyun/jupyter_home/yolo8n_tank_image/extest_images'

model = YOLO(model_path)

results = model.predict(
    source=test_img_dir,
    save=True,
    save_txt=True,
    imgsz=640,
    conf=0.25
)

print("✅ 예측 완료")

In [ ]:
from IPython.display import display, Image
import glob

# 예측 이미지 저장된 폴더 경로
predict_dir = './runs/detect/predict/'

# 예측 결과 이미지들 불러오기 (확장자 jpg/png)
pred_images = glob.glob(predict_dir + '*.jpg') + glob.glob(predict_dir + '*.png')

# 결과 이미지 5개까지 출력
for img_path in pred_images[:5]:
    display(Image(filename=img_path))

In [ ]:
data_yaml_40_content = """
train: /home/jonggyun/jupyter_home/yolo8n_tank_image/yolov8n40/images
val: /home/jonggyun/jupyter_home/yolo8n_tank_image/yolov8n40/images

nc: 3
names: ['E_Tank', 'Car', 'Human']
"""

with open("/home/jonggyun/jupyter_home/yolo8n_tank_image/yolov8n40/data_40.yaml", "w") as f:
    f.write(data_yaml_40_content)

print("✅ 40% 전용 data.yaml 생성 완료")

In [ ]:
from ultralytics import YOLO

# 기존 학습된 60% 모델 불러오기
model = YOLO('/home/jonggyun/jupyter_home/yolo8n_tank_image/multi_class_yolov8/weights/best.pt')

# 40% 데이터로 이어서 재학습 (fine-tuning)
model.train(
    data='/home/jonggyun/jupyter_home/yolo8n_tank_image/yolov8n40/data_40.yaml',
    epochs=100,        # 너무 크게 주지 않고 적절히 (예: 50~100)
    imgsz=640,
    batch=16,
    name='yolov8n_finetune_40',
    project='/home/jonggyun/jupyter_home/yolo8n_tank_image',
    augment=True,
    lr0=0.0001,         # fine-tune이므로 낮은 학습률 권장
    resume=False        # 기존 모델에서 이어서, 그러나 새 세션으로 시작
)

print("✅ 40% 데이터로 기존 모델을 fine-tuning 완료")